In [1]:
import pandas as pandas_library
import numpy as numpy_library

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

import joblib

In [2]:
dataframe = pandas_library.read_csv("HR_comma_sep.csv")

print(dataframe.shape)
dataframe.head()

(14999, 10)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,dept,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


In [3]:
dataframe.info()
dataframe.isnull().sum()
dataframe['left'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   satisfaction_level     14999 non-null  float64
 1   last_evaluation        14999 non-null  float64
 2   number_project         14999 non-null  int64  
 3   average_montly_hours   14999 non-null  int64  
 4   time_spend_company     14999 non-null  int64  
 5   Work_accident          14999 non-null  int64  
 6   left                   14999 non-null  int64  
 7   promotion_last_5years  14999 non-null  int64  
 8   dept                   14999 non-null  object 
 9   salary                 14999 non-null  object 
dtypes: float64(2), int64(6), object(2)
memory usage: 1.1+ MB


left
0    11428
1     3571
Name: count, dtype: int64

In [4]:
target_column = "left"

input_features = dataframe.drop(columns=[target_column])
target_variable = dataframe[target_column]

In [5]:
numerical_columns = input_features.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_columns = input_features.select_dtypes(include=['object']).columns.tolist()

print("Numerical:", numerical_columns)
print("Categorical:", categorical_columns)

Numerical: ['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company', 'Work_accident', 'promotion_last_5years']
Categorical: ['dept', 'salary']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    input_features,
    target_variable,
    test_size=0.2,
    random_state=42,
    stratify=target_variable
)

In [7]:
numerical_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ]
)

In [8]:
model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

In [9]:
model_pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
predictions = model_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))

Accuracy: 0.99

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      2286
           1       0.99      0.96      0.98       714

    accuracy                           0.99      3000
   macro avg       0.99      0.98      0.99      3000
weighted avg       0.99      0.99      0.99      3000


Confusion Matrix:
 [[2282    4]
 [  26  688]]


In [11]:
trained_model = model_pipeline.named_steps["model"]
print(trained_model.feature_importances_)

[0.26811992 0.13372886 0.15222128 0.15161071 0.23782373 0.01257909
 0.00215539 0.00168148 0.00185818 0.00192193 0.00234331 0.00169416
 0.00133333 0.0009887  0.00412778 0.00336931 0.00466492 0.0061823
 0.00750546 0.00409017]


In [12]:
joblib.dump(model_pipeline, "employee_attrition_pipeline.pkl")
print("Pipeline saved successfully")

Pipeline saved successfully


In [13]:
loaded_pipeline = joblib.load("employee_attrition_pipeline.pkl")

sample_prediction = loaded_pipeline.predict(X_test.iloc[[0]])
print("Prediction:", sample_prediction)

Prediction: [0]
